# Notification and Automation notebook

Notification and Automation is the event-driven layer of the platform, built on Knative. Events from various sources - a GitHub or GitLab webhook, a Kubernetes API event, a STAC item being registered in [Data Access](../data-access/data-access.ipynb) - get converted into [CloudEvents](https://cloudevents.io/) and routed through a broker to whatever's subscribed.

This notebook checks the BB is up, looks at events that are already flowing with no setup required, sends some webhooks through (GitHub and GitLab, including per-project routing), then shows a *different* Building Block's activity (a STAC registration in Data Access) arriving here with no custom integration code, and finally wires up a function to react to a real event rather than a synthetic one.

## Setup

In [ ]:
import hashlib
import hmac
import json
import os
import sys
import uuid
from time import sleep, time

import requests

sys.path.append('../')
from modules.helpers import load_eoepca_state, get_access_token, test_cell, test_results

Load the eoepca state environment variables.

In [ ]:
load_eoepca_state()

In [ ]:
platform_domain = os.environ["INGRESS_HOST"]
http_scheme = os.environ["HTTP_SCHEME"]
github_webhook_secret = os.environ["NA_GITHUB_WEBHOOK_SECRET"]
gitlab_webhook_secret = os.environ["NA_GITLAB_WEBHOOK_SECRET"]

webhook_source_url = f"{http_scheme}://webhooks.notifications.{platform_domain}"
cloudevents_player_url = f"{http_scheme}://cloudevents-player.notifications.{platform_domain}"
eoapi_url = f"{http_scheme}://eoapi.{platform_domain}"

log_output_file = "notification-automation_log.json"

## Check the Building Block is up

The webhook source and CloudEvents player each get their own ingress. A quick health check on both confirms the chart deployed correctly before going further.

In [ ]:
# endpoints_healthy
player_response = requests.get(cloudevents_player_url, timeout=10)
webhook_health_response = requests.get(f"{webhook_source_url}/health", timeout=10)

print(f"CloudEvents Player: {player_response.status_code}")
print(f"Webhook source health: {webhook_health_response.status_code} {webhook_health_response.text}")

assert player_response.status_code == 200
assert webhook_health_response.status_code == 200

## Watching events flow: the CloudEvents Player

The chart deploys a `default` Broker with a Trigger already subscribing the CloudEvents Player to it, and an API Server Source turning Kubernetes `Event` objects in the `notifications` namespace into CloudEvents on that same broker. So the player already shows real traffic with no setup on our part - a general-purpose "what just happened in this namespace" view, essentially for free. Open `cloudevents_player_url` in a browser to watch it live, or pull the same data over its REST API below.

In [ ]:
def fetch_events(size=200):
    response = requests.get(
        f"{cloudevents_player_url}/messages",
        params={"page": 0, "size": size},
        timeout=10,
    )
    response.raise_for_status()
    return response.json()

In [ ]:
# api_server_source_events
events = fetch_events()
print(f"CloudEvents Player has {len(events)} events recorded")
for event in events[:3]:
    print(f"  {event['eventType']} from {event['source']}")

assert len(events) > 0
assert any(event["eventType"].startswith("dev.knative.apiserver.") for event in events)

## Turning a GitHub webhook into a CloudEvent

The webhook source's `/github` endpoint is what a real repository's webhook settings would point at, using `NA_GITHUB_WEBHOOK_SECRET` (from `~/.eoepca/state`) as the secret. GitHub signs every delivery with `X-Hub-Signature-256`, an HMAC-SHA256 of the raw body using that secret, and the webhook source verifies it before accepting anything. We simulate a delivery here rather than requiring a public repository, using the same signature scheme.

In [ ]:
def signed_github_request(payload: dict, secret: str):
    body = json.dumps(payload).encode()
    signature = "sha256=" + hmac.new(secret.encode(), body, hashlib.sha256).hexdigest()
    headers = {
        "Content-Type": "application/json",
        "X-GitHub-Event": "ping",
        "X-GitHub-Delivery": str(uuid.uuid4()),
        "X-Hub-Signature-256": signature,
    }
    return body, headers

A request signed with the wrong secret is rejected outright - it never reaches the broker.

In [ ]:
# webhook_signature_rejected
body, headers = signed_github_request({"zen": "wrong secret"}, "not-the-real-secret")
response = requests.post(f"{webhook_source_url}/github", data=body, headers=headers, timeout=10)

print(f"Status: {response.status_code} {response.text}")
assert response.status_code == 401

A correctly-signed delivery is accepted immediately (`202`) and shows up moments later in the CloudEvents Player as an `org.eoepca.webhook.github.ping` event.

In [ ]:
# webhook_event_delivered
github_marker = str(uuid.uuid4())
payload = {"zen": "Automation over integration.", "hook_id": 1, "marker": github_marker}
body, headers = signed_github_request(payload, github_webhook_secret)

response = requests.post(f"{webhook_source_url}/github", data=body, headers=headers, timeout=10)
print(f"Delivery accepted: {response.status_code} {response.json()}")
assert response.status_code == 202

deadline = time() + 30
delivered = None
while time() < deadline and delivered is None:
    for event in fetch_events():
        if event.get("data", {}).get("marker") == github_marker:
            delivered = event
            break
    if delivered is None:
        sleep(2)

print(f"Delivered event: {json.dumps(delivered, indent=2) if delivered else 'not seen within 30s'}")
assert delivered is not None
assert delivered["eventType"] == "org.eoepca.webhook.github.ping"

## Turning a GitLab webhook into a CloudEvent

GitLab uses a different scheme to GitHub - a plain secret token in `X-Gitlab-Token`, compared directly rather than hashed, using `NA_GITLAB_WEBHOOK_SECRET`. Same webhook source, same broker, same player - only the header and the resulting `eventType` (`org.eoepca.webhook.gitlab.*`) differ.

In [ ]:
def gitlab_headers(secret: str, event: str = "Push Hook"):
    return {
        "Content-Type": "application/json",
        "X-Gitlab-Event": event,
        "X-Gitlab-Token": secret,
    }

In [ ]:
# gitlab_webhook_delivered
gitlab_marker = str(uuid.uuid4())
payload = {"project": {"web_url": "https://gitlab.com/EOEPCA/deployment-guide"}, "marker": gitlab_marker}
response = requests.post(
    f"{webhook_source_url}/gitlab",
    json=payload,
    headers=gitlab_headers(gitlab_webhook_secret),
    timeout=10,
)
print(f"Delivery accepted: {response.status_code} {response.text}")
assert response.status_code == 202

deadline = time() + 30
delivered = None
while time() < deadline and delivered is None:
    for event in fetch_events():
        if event.get("data", {}).get("marker") == gitlab_marker:
            delivered = event
            break
    if delivered is None:
        sleep(2)

print(f"Delivered event: {json.dumps(delivered, indent=2) if delivered else 'not seen within 30s'}")
assert delivered is not None
assert delivered["eventType"] == "org.eoepca.webhook.gitlab.push_hook"

## Everything below needs `kubectl`

Routing webhooks per project, creating an isolated broker, and wiring your own function all need direct cluster access to apply manifests - none of it is exposed through the webhook source's own API. These sections are skipped automatically (and aren't part of the automated smoketest) if `kubectl` isn't on the `PATH` or isn't configured against the cluster.

In [ ]:
import shutil
import subprocess

kubectl_available = shutil.which("kubectl") is not None
print(f"kubectl available: {kubectl_available}")

### Routing webhooks from multiple projects

The webhook source reads a `PROJECTS_CONFIG` ConfigMap on startup, mapping project names to their own secrets, so different repositories don't have to share `NA_GITHUB_WEBHOOK_SECRET`. The resulting CloudEvent's `subject` is set to the project name, which is useful for routing different repositories to different Triggers. The ConfigMap is only read at startup - applying it alone has no effect until the pod restarts.

In [ ]:
project_name = "notebook-demo-project"
project_secret = "a-different-secret-for-this-project"

projects_configmap_yaml = f"""\
apiVersion: v1
kind: ConfigMap
metadata:
  name: notification-automation-webhook-source
  namespace: notifications
data:
  projects.json: |
    {{"{project_name}": {{"github_secret": "{project_secret}"}}}}
"""

if kubectl_available:
    try:
        subprocess.run(["kubectl", "apply", "-f", "-"], input=projects_configmap_yaml, text=True, check=True)
        subprocess.run(
            ["kubectl", "rollout", "restart", "deployment/notification-automation-webhook-source", "-n", "notifications"],
            check=True,
        )
        subprocess.run(
            ["kubectl", "rollout", "status", "deployment/notification-automation-webhook-source", "-n", "notifications", "--timeout=60s"],
            check=True,
        )

        # rollout status reports Ready slightly before the old pod is fully removed from
        # the Service, so the first request or two can still land on it and 404 - retry
        # rather than treat that as a hard failure.
        body, headers = signed_github_request({"marker": "project-routing-demo"}, project_secret)
        for attempt in range(5):
            response = requests.post(f"{webhook_source_url}/{project_name}/github", data=body, headers=headers, timeout=10)
            if response.status_code == 202:
                break
            sleep(2)
        print(f"Project-specific delivery: {response.status_code} {response.text}")
        assert response.status_code == 202
    finally:
        subprocess.run(["kubectl", "delete", "configmap", "notification-automation-webhook-source", "-n", "notifications", "--ignore-not-found"])
        subprocess.run(["kubectl", "rollout", "restart", "deployment/notification-automation-webhook-source", "-n", "notifications"])
else:
    print("kubectl not available on PATH - skipping.")

## Watching a STAC registration happen

[Data Access](../data-access/data-access.ipynb) can emit a CloudEvent every time a STAC item changes, via its own `eoapi-notifier` component listening on pgSTAC's `pgstac_items_change` channel. This is genuinely independent of this BB - the two are wired together only by both pointing at the same broker (`ENABLE_EOAPI_NOTIFIER=yes` in Data Access's own configuration). No custom integration code on either side, just two Building Blocks agreeing on where to send events.

This section requires Data Access to actually be deployed with that flag set - it checks first and skips with a note if it isn't, since this notebook should still work when Notification and Automation is the only thing deployed.

In [ ]:
try:
    eoapi_response = requests.get(f"{eoapi_url}/stac", timeout=5)
    eoapi_available = eoapi_response.status_code < 500
except requests.RequestException:
    eoapi_available = False

print(f"Data Access reachable at {eoapi_url}: {eoapi_available}")
if not eoapi_available:
    print("Skipping - deploy Data Access with ENABLE_EOAPI_NOTIFIER=yes to see this section in action.")

In [ ]:
if eoapi_available:
    stac_headers = {"Content-Type": "application/json"}
    id_prefix = ""

    if os.environ.get("DATA_ACCESS_ENABLE_IAM") == "yes":
        access_token = get_access_token(
            os.environ["KEYCLOAK_TEST_USER"],
            os.environ["KEYCLOAK_TEST_PASSWORD"],
            os.environ["EOAPI_CLIENT_ID"],
        )
        stac_headers["Authorization"] = f"Bearer {access_token}"
        id_prefix = f"{os.environ['KEYCLOAK_TEST_USER']}."

    collection_id = f"{id_prefix}na-notebook-demo"
    item_id = f"na-demo-item-{uuid.uuid4().hex[:8]}"

    collection = {
        "id": collection_id, "type": "Collection", "stac_version": "1.0.0",
        "description": "Created by the Notification and Automation notebook",
        "license": "proprietary",
        "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [[None, None]]}},
        "links": [],
    }
    collection_response = requests.post(f"{eoapi_url}/stac/collections", json=collection, headers=stac_headers, timeout=10)
    print(f"Collection: {collection_response.status_code}")
    assert collection_response.status_code in (200, 201, 409)

    item = {
        "id": item_id, "type": "Feature", "stac_version": "1.0.0", "collection": collection_id,
        "geometry": {"type": "Point", "coordinates": [0, 0]}, "bbox": [0, 0, 0, 0],
        "properties": {"datetime": "2026-01-01T00:00:00Z"}, "links": [], "assets": {},
    }
    item_response = requests.post(f"{eoapi_url}/stac/collections/{collection_id}/items", json=item, headers=stac_headers, timeout=10)
    print(f"Item: {item_response.status_code}")
    assert item_response.status_code == 201

    deadline = time() + 30
    stac_event = None
    while time() < deadline and stac_event is None:
        for event in fetch_events():
            if event.get("source") == "/eoapi/pgstac" and event.get("subject") == item_id:
                stac_event = event
                break
        if stac_event is None:
            sleep(2)

    print(f"CloudEvent from Data Access: {json.dumps(stac_event, indent=2) if stac_event else 'not seen within 30s'}")
    assert stac_event is not None
    assert stac_event["eventType"] == "org.ogc.api.collection.item.create"

### Creating an isolated broker

`default` (used throughout above) already carries webhook, API-server and Data Access events. Create your own broker when a separate event space is needed instead - for example so your own Triggers don't match unrelated platform events.

In [ ]:
isolated_broker_yaml = """\
apiVersion: eventing.knative.dev/v1
kind: Broker
metadata:
  name: notebook-demo
  namespace: notifications
"""

if kubectl_available:
    try:
        subprocess.run(["kubectl", "apply", "-f", "-"], input=isolated_broker_yaml, text=True, check=True)
        subprocess.run(
            ["kubectl", "wait", "--for=condition=Ready", "broker/notebook-demo", "-n", "notifications", "--timeout=60s"],
            check=True,
        )
        print("Broker 'notebook-demo' is Ready - independent of the 'default' broker used above.")
    finally:
        subprocess.run(["kubectl", "delete", "broker", "notebook-demo", "-n", "notifications", "--ignore-not-found"])
else:
    print("kubectl not available on PATH - skipping.")

### Reacting to events with your own function

This is the other half of the BB: subscribe a Knative Service to `default` with a Trigger filtered on a specific event type, and it fires whenever a matching event arrives. Rather than a synthetic event, this reacts to the same GitHub webhook delivery used earlier - deploy the standard Knative sample as a disposable function, subscribe it to `org.eoepca.webhook.github.ping`, resend a signed webhook, and watch the function pick it up.

In [ ]:
hello_function_yaml = """\
apiVersion: serving.knative.dev/v1
kind: Service
metadata:
  name: hello-function
  namespace: notifications
spec:
  template:
    spec:
      containers:
        - image: gcr.io/knative-samples/helloworld-go
          env:
            - name: TARGET
              value: "EOEPCA Platform"
"""

hello_function_trigger_yaml = """\
apiVersion: eventing.knative.dev/v1
kind: Trigger
metadata:
  name: hello-function-trigger
  namespace: notifications
spec:
  broker: default
  filter:
    attributes:
      type: org.eoepca.webhook.github.ping
  subscriber:
    ref:
      apiVersion: serving.knative.dev/v1
      kind: Service
      name: hello-function
"""

def kubectl_cleanup():
    subprocess.run(["kubectl", "delete", "trigger", "hello-function-trigger", "-n", "notifications", "--ignore-not-found"])
    subprocess.run(["kubectl", "delete", "ksvc", "hello-function", "-n", "notifications", "--ignore-not-found"])

if kubectl_available:
    try:
        subprocess.run(["kubectl", "apply", "-f", "-"], input=hello_function_yaml, text=True, check=True)
        subprocess.run(["kubectl", "wait", "--for=condition=Ready", "ksvc/hello-function", "-n", "notifications", "--timeout=180s"], check=True)
        subprocess.run(["kubectl", "apply", "-f", "-"], input=hello_function_trigger_yaml, text=True, check=True)
        # the Trigger needs to be Ready before an event fired immediately after will
        # reliably be routed to it - otherwise it's simply not subscribed yet, which
        # retries don't fix.
        subprocess.run(["kubectl", "wait", "--for=condition=Ready", "trigger/hello-function-trigger", "-n", "notifications", "--timeout=60s"], check=True)
    except Exception:
        kubectl_cleanup()
        raise

Resend a signed GitHub webhook (the same helper as above) and check the function's own logs for it - no synthetic curl event needed, it reacts to the real thing:

In [ ]:
if kubectl_available:
    try:
        body, headers = signed_github_request({"zen": "Trigger the function", "hook_id": 2}, github_webhook_secret)
        try:
            response = requests.post(f"{webhook_source_url}/github", data=body, headers=headers, timeout=35)
            print(f"Webhook delivery: {response.status_code}")
        except requests.RequestException as e:
            print(f"Webhook delivery request itself was slow/failed ({e}) - checking whether it still got through.")

        deadline = time() + 90
        logs = None
        while time() < deadline:
            logs = subprocess.run(
                ["kubectl", "logs", "-n", "notifications", "-l", "serving.knative.dev/service=hello-function", "-c", "user-container", "--tail=10"],
                capture_output=True, text=True,
            )
            if "received a request" in logs.stdout:
                break
            sleep(3)

        print(logs.stdout if logs else "(no logs captured)")
        assert logs is not None and "received a request" in logs.stdout
    finally:
        kubectl_cleanup()

You should see a `helloworld: received a request` line - the webhook delivered above didn't just land in the CloudEvents Player, it drove a real piece of automation.

## Optional: Slack and email

Two more subscribers ship with (or alongside) this BB, not exercised here since both require real external credentials this notebook can't assume:

- **Slack**: [`send-notification-to-slack`](https://github.com/EOEPCA/send-notification-to-slack), a Knative function that posts any CloudEvent to a Slack channel - needs a real Slack Incoming Webhook URL.
- **Email**: the chart's own emailer sink (`NA_ENABLE_EMAILER=yes`) - needs a real SMTP relay.

See the deployment guide's ["Optional: notify Slack"](../../../docs/building-blocks/notification-automation.md#optional-notify-slack) and ["Optional: email a CloudEvent"](../../../docs/building-blocks/notification-automation.md#optional-email-a-cloudevent) sections for the exact commands - both are just a Trigger away from what's already been demonstrated above.

## Results

In [ ]:
if test_results:
    for test, result in test_results.items():
        print(f"{test}: {result['status']} - {result['message']}")
    json.dump(test_results, open(log_output_file, "w"), indent=2)